In [ ]:
import os
from datasets import load_dataset
from huggingface_hub import login
import json
from train_token_classifier import run_training, TrainConfig
from pathlib import Path
import pandas as pd

HF_TOKEN = ""
REPO_ID = "mandal437/bert-token-detector-bio"
WORKDIR = Path.cwd()

os.environ["HF_TOKEN"] = HF_TOKEN.strip()
login(token=HF_TOKEN.strip(), add_to_git_credential=False)

ds = load_dataset(REPO_ID, token=True)
print(ds)
print(len(ds["train"]), len(ds["validation"]), len(ds["test"]))

/home/jupyter/.local/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2026-05-06 16:07:40.268260: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 225608
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 26330
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 28317
    })
})
225608 26330 28317


In [2]:
from collections import Counter

def analyze_window_label_patterns(split_ds, split_name="train"):
    patterns = Counter()

    for ex in split_ds:
        labels = [x for x in ex["labels"] if x != -100]
        uniq = set(labels)

        if uniq == {0}:
            patterns["only_O"] += 1
        elif uniq == {1, 2}:
            patterns["only_AI_no_O"] += 1
        elif uniq == {0, 1, 2}:
            patterns["mixed_O_B_I"] += 1
        elif uniq == {0, 2}:
            patterns["O_and_I_only"] += 1
        elif uniq == {1}:
            patterns["only_B"] += 1
        elif uniq == {2}:
            patterns["only_I"] += 1
        else:
            patterns[str(sorted(list(uniq)))] += 1

    print(f"\n=== {split_name} window pattern stats ===")
    total = sum(patterns.values())
    for k, v in patterns.most_common():
        print(f"{k:20s} {v:8d}  ({v/total:.4%})")

# пример
analyze_window_label_patterns(ds["train"], "train")
analyze_window_label_patterns(ds["validation"], "validation")
analyze_window_label_patterns(ds["test"], "test")



=== train window pattern stats ===
only_O                 100244  (44.4328%)
only_AI_no_O            90112  (39.9418%)
mixed_O_B_I             35169  (15.5885%)
[0, 1]                     83  (0.0368%)

=== validation window pattern stats ===
only_O                  11740  (44.5879%)
only_AI_no_O            10449  (39.6848%)
mixed_O_B_I              4128  (15.6779%)
[0, 1]                     13  (0.0494%)

=== test window pattern stats ===
only_O                  12408  (43.8182%)
only_AI_no_O            11398  (40.2514%)
mixed_O_B_I              4496  (15.8774%)
[0, 1]                     15  (0.0530%)


In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("microsoft/mdeberta-v3-base", use_fast=True)

def show_mixed_examples_with_tokens(split_ds, n=3, max_items=80):
    found = 0

    for i, ex in enumerate(split_ds):
        labels = [x for x in ex["labels"] if x != -100]
        uniq = set(labels)

        if uniq == {0, 1, 2}:
            tokens = tokenizer.convert_ids_to_tokens(ex["input_ids"])

            print(f"\n===== EXAMPLE #{found+1} | dataset index = {i} =====")
            for tok, lab in zip(tokens[:max_items], ex["labels"][:max_items]):
                print(f"{tok:20s} -> {lab}")
            
            found += 1
            if found >= n:
                break

show_mixed_examples_with_tokens(ds["train"], n=3, max_items=100)

/home/jupyter/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 9543f91d-fc63-46c5-abbd-62e01485b92e)')' thrown while requesting HEAD https://huggingface.co/microsoft/mdeberta-v3-base/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
/home/jupyter/.local/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:550: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted


===== EXAMPLE #1 | dataset index = 4727 =====
[CLS]                -> -100
-                    -> 0
▁                    -> 0
[UNK]                -> 0
,                    -> 0
▁2014.               -> 0
▁                    -> 0
[UNK]                -> 0
.                    -> 0
▁262                 -> 0
].                   -> 0
▁                    -> 0
[UNK]                -> 0
▁                    -> 0
[UNK]                -> 0
▁[                   -> 0
[UNK]                -> 0
,                    -> 0
▁2013.               -> 0
▁                    -> 0
[UNK]                -> 0
.                    -> 0
▁                    -> 0
69]                  -> 0
.                    -> 0
▁                    -> 0
[UNK]                -> 0
▁                    -> 0
[UNK]                -> 0
▁                    -> 0
[UNK]                -> 0
▁(188                -> 0
2/8                  -> 0
4                    -> 0
[UNK]                -> 0
▁                    -> 0
[UNK]         

In [ ]:
BLOCK1_OUT = WORKDIR / "token_detector_block1"

base_config = dict(
    hf_dataset_dir="",
    bio_data_dir="",
    model_name="microsoft/mdeberta-v3-base",
    train_mode="head_only",                # фиксируем head_only для чистоты сравнения loss'ов
    partial_unfreeze_last_n_layers=2,     # не используется, но оставим
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    max_grad_norm=1.0,
    logging_steps=50,
    save_total_limit=1,
    dataloader_num_workers=2,
    seed=42,
    fp16=False,
    bf16=False,
    cpu_only=False,
    gradient_checkpointing=False,
    group_by_length=True,
    eval_strategy="epoch",
    save_strategy="no",
    eval_steps=500,
    save_steps=500,
    load_best_model_at_end=False,
    # Sampler отключаем для изоляции эффекта лосса
    use_weighted_sampler=False,
    sampler_human_weight=1.0,
    sampler_ai_weight=1.5,
    sampler_bai_weight=4.0,
)

# Определяем 4 конфигурации лосса
experiments = {
    "1a_CE_baseline": {
        "use_focal_loss": False,
        "use_class_balanced_alpha": False,
        "manual_alpha": None,
        "focal_gamma": 1.5,    # не используется
    },
    "1b_Weighted_CE": {
        "use_focal_loss": False,
        "use_class_balanced_alpha": False,
        "manual_alpha": [1.0, 3.0, 5.0],  # O, B-AI, I-AI – можно менять для подбора
        "focal_gamma": 1.5,
    },
    "1c_Focal": {
        "use_focal_loss": True,
        "use_class_balanced_alpha": False,
        "manual_alpha": None,
        "focal_gamma": 1.5,
    },
    "1d_CB_Focal": {
        "use_focal_loss": True,
        "use_class_balanced_alpha": True,
        "effective_num_beta": 0.9999,
        "alpha_clip_max": 10.0,
        "manual_alpha": None,
        "focal_gamma": 1.5,
    },
}

results = []

for exp_name, loss_params in experiments.items():
    output_dir = BLOCK1_OUT / exp_name
    print(f"\n{'='*60}")
    print(f"Running {exp_name} -> {output_dir}")
    print(f"{'='*60}")
    
    # Собираем полный конфиг
    cfg = TrainConfig(
        **base_config,
        output_dir=str(output_dir),
        **loss_params
    )
    
    # Запускаем обучение
    run_training(cfg, dataset_override=ds)
    
    # Читаем сохранённые метрики на тесте
    test_metrics_path = output_dir / "test_metrics.json"
    if test_metrics_path.exists():
        with open(test_metrics_path) as f:
            test_metrics = json.load(f)
        # Извлекаем только нужные (префикс "test_" добавлен Trainer)
        row = {
            "experiment": exp_name,
            "seqeval_f1": test_metrics.get("test_seqeval_f1", None),
            "ai_token_f1": test_metrics.get("test_ai_token_f1", None),
            "class_B-AI_f1": test_metrics.get("test_class_B-AI_f1", None),
            "token_f1_macro": test_metrics.get("test_token_f1_macro", None),
        }
        results.append(row)
        print(f"Test metrics: {row}")
    else:
        print(f"WARNING: {test_metrics_path} not found!")
        results.append({"experiment": exp_name, "seqeval_f1": None})

# Сводная таблица
df = pd.DataFrame(results)
print("\n\n=== BLOCK 1 RESULTS ===")
print(df.to_string(index=False))

# Определяем лучший по seqeval_f1 (главная метрика BIO)
if df["seqeval_f1"].notna().any():
    best = df.loc[df["seqeval_f1"].idxmax()]
    print(f"\nBest loss function based on test_seqeval_f1: {best['experiment']} "
          f"with seqeval_f1 = {best['seqeval_f1']:.4f}")
else:
    print("No valid results to compare.")

 72%|███████▏  | 20400/28201 [1:02:09<23:46,  5.47it/s]

{'loss': 0.6138, 'grad_norm': 5.597281455993652, 'learning_rate': 8.555604299188417e-05, 'epoch': 0.72}


 73%|███████▎  | 20450/28201 [1:02:18<23:20,  5.54it/s]

{'loss': 0.6251, 'grad_norm': 3.339423179626465, 'learning_rate': 8.500767712217591e-05, 'epoch': 0.73}


 73%|███████▎  | 20500/28201 [1:02:27<24:13,  5.30it/s]

...

 45%|████▌     | 12700/28201 [39:16<46:40,  5.54it/s]

{'loss': 0.2236, 'grad_norm': 5.729939937591553, 'learning_rate': 0.00017000438692695764, 'epoch': 0.45}


 45%|████▌     | 12750/28201 [39:25<46:59,  5.48it/s]

{'loss': 0.2254, 'grad_norm': 0.6773124933242798, 'learning_rate': 0.00016945602105724938, 'epoch': 0.45}


 45%|████▌     | 12790/28201 [39:32<47:54,  5.36it/s]

In [1]:
import json
from pathlib import Path
import pandas as pd

BLOCK1_DIR = Path("token_detector_block1")
experiments = ["1a_CE_baseline", "1b_Weighted_CE", "1c_Focal", "1d_CB_Focal"]

results = []
for exp in experiments:
    metrics_path = BLOCK1_DIR / exp / "test_metrics.json"
    if not metrics_path.exists():
        print(f"WARNING: {metrics_path} not found")
        continue
    with open(metrics_path) as f:
        metrics = json.load(f)
    results.append({
        "experiment": exp,
        # с названиями, которые сохраняет твой Trainer (test_seqeval_f1 и т.д.)
        "test_seqeval_f1": metrics.get("test_seqeval_f1", 0),
        "test_ai_token_f1": metrics.get("test_ai_token_f1", 0),
        "test_class_B-AI_f1": metrics.get("test_class_B-AI_f1", 0),
        "test_token_f1_macro": metrics.get("test_token_f1_macro", 0),
        "test_seqeval_precision": metrics.get("test_seqeval_precision", 0),
        "test_seqeval_recall": metrics.get("test_seqeval_recall", 0),
    })

df = pd.DataFrame(results).sort_values("test_seqeval_f1", ascending=False)
print(df.to_string(index=False))

best = df.iloc[0]
print(f"\n🏆 Best loss function: {best['experiment']} with test_seqeval_f1 = {best['test_seqeval_f1']:.4f}")

    experiment  test_seqeval_f1  test_ai_token_f1  test_class_B-AI_f1  test_token_f1_macro  test_seqeval_precision  test_seqeval_recall
1b_Weighted_CE         0.009088          0.664766            0.546897             0.499318                0.004683             0.153435
      1c_Focal         0.000086          0.681238            0.526543             0.610193                0.000043             0.003206
   1d_CB_Focal         0.000086          0.681237            0.526543             0.610193                0.000043             0.003206
1a_CE_baseline         0.000084          0.681165            0.528460             0.612476                0.000042             0.003143

🏆 Best loss function: 1b_Weighted_CE with test_seqeval_f1 = 0.0091


In [ ]:
import json
from pathlib import Path
import pandas as pd

from train_token_classifier import run_training, TrainConfig

WORKDIR = Path.cwd()

BLOCK2_OUT = WORKDIR / "token_detector_block2"
BLOCK2_OUT.mkdir(parents=True, exist_ok=True)

BEST_STAGE1_MODEL = WORKDIR / "token_detector_block1" / "1b_Weighted_CE" / "final_model"

print("WORKDIR:", WORKDIR)
print("BLOCK2_OUT:", BLOCK2_OUT)
print("BEST_STAGE1_MODEL exists:", BEST_STAGE1_MODEL.exists())

base_config = dict(
    hf_dataset_dir="",
    bio_data_dir="",

    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,

    weight_decay=0.01,
    max_grad_norm=1.0,
    warmup_steps=0,

    logging_steps=50,
    save_total_limit=1,
    dataloader_num_workers=2,
    seed=42,

    fp16=False,
    bf16=False,
    cpu_only=False,
    gradient_checkpointing=False,
    group_by_length=True,
    use_focal_loss=False,
    use_class_balanced_alpha=False,
    manual_alpha=[1.0, 3.0, 5.0],
    effective_num_beta=0.9999,
    alpha_clip_max=10.0,
    focal_gamma=1.5,

    use_weighted_sampler=False,
    sampler_human_weight=1.0,
    sampler_ai_weight=1.5,
    sampler_bai_weight=4.0,

    eval_strategy="epoch",
    save_strategy="no",
    eval_steps=500,
    save_steps=500,
    load_best_model_at_end=False,
)

experiments = {
    "2a_head_only": {
        "model_name": "microsoft/mdeberta-v3-base",
        "train_mode": "head_only",
        "partial_unfreeze_last_n_layers": 2,
        "learning_rate": 3e-4,
        "warmup_ratio": 0.03,
    },
    "2b_partial_unfreeze_last_2": {
        "model_name": str(BEST_STAGE1_MODEL),
        "train_mode": "partial_unfreeze",
        "partial_unfreeze_last_n_layers": 2,
        "learning_rate": 3e-6,
        "warmup_ratio": 0.02,
        "max_grad_norm": 0.5,
    },
    "2c_partial_unfreeze_last_4": {
        "model_name": str(BEST_STAGE1_MODEL),
        "train_mode": "partial_unfreeze",
        "partial_unfreeze_last_n_layers": 4,
        "learning_rate": 2e-6,
        "warmup_ratio": 0.02,
        "max_grad_norm": 0.5,
    },
    "2d_full_finetune": {
        "model_name": str(BEST_STAGE1_MODEL),
        "train_mode": "full",
        "partial_unfreeze_last_n_layers": 2,  # не используется, но поле должно быть
        "learning_rate": 1e-6,
        "warmup_ratio": 0.02,
        "max_grad_norm": 0.5,
    },
}

WORKDIR: /home/jupyter/project/ai_detector_diploma/src/token_detector
BLOCK2_OUT: /home/jupyter/project/ai_detector_diploma/src/token_detector/token_detector_block2
BEST_STAGE1_MODEL exists: True


In [ ]:
results = []

for exp_name, exp_params in experiments.items():
    output_dir = BLOCK2_OUT / exp_name

    print(f"\n{'=' * 80}")
    print(f"RUNNING EXPERIMENT: {exp_name}")
    print(f"OUTPUT DIR: {output_dir}")
    print(f"{'=' * 80}")

    cfg_dict = {**base_config, **exp_params, "output_dir": str(output_dir)}
    cfg = TrainConfig(**cfg_dict)

    run_training(cfg, dataset_override=ds)

    test_metrics_path = output_dir / "test_metrics.json"
    val_metrics_path = output_dir / "val_metrics.json"
    run_metadata_path = output_dir / "run_metadata.json"

    row = {
        "experiment": exp_name,
        "output_dir": str(output_dir),
    }

    if run_metadata_path.exists():
        with open(run_metadata_path, "r", encoding="utf-8") as f:
            run_meta = json.load(f)

        row["train_mode"] = run_meta["run_config"].get("train_mode")
        row["partial_unfreeze_last_n_layers"] = run_meta["run_config"].get("partial_unfreeze_last_n_layers")
        row["learning_rate"] = run_meta["run_config"].get("learning_rate")
        row["trainable_params"] = run_meta.get("trainable_params")

    if val_metrics_path.exists():
        with open(val_metrics_path, "r", encoding="utf-8") as f:
            val_metrics = json.load(f)

        row["val_seqeval_f1"] = val_metrics.get("val_seqeval_f1", None)
        row["val_ai_token_f1"] = val_metrics.get("val_ai_token_f1", None)
        row["val_class_B-AI_f1"] = val_metrics.get("val_class_B-AI_f1", None)
        row["val_class_O_recall"] = val_metrics.get("val_class_O_recall", None)
        row["val_token_f1_macro"] = val_metrics.get("val_token_f1_macro", None)

    if test_metrics_path.exists():
        with open(test_metrics_path, "r", encoding="utf-8") as f:
            test_metrics = json.load(f)

        row["test_seqeval_f1"] = test_metrics.get("test_seqeval_f1", None)
        row["test_ai_token_f1"] = test_metrics.get("test_ai_token_f1", None)
        row["test_class_B-AI_f1"] = test_metrics.get("test_class_B-AI_f1", None)
        row["test_class_O_recall"] = test_metrics.get("test_class_O_recall", None)
        row["test_token_f1_macro"] = test_metrics.get("test_token_f1_macro", None)

    results.append(row)

# сохраняем сырые результаты блока
results_path = BLOCK2_OUT / "block2_results.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\nSaved raw results to: {results_path}")



                                                       
100%|██████████| 28201/28201 [3:37:07<00:00,  2.16it/s]A

{'eval_loss': 0.13609594106674194, 'eval_token_precision_macro': 0.9167078092608193, 'eval_token_recall_macro': 0.9322101967806553, 'eval_token_f1_macro': 0.9238050349567709, 'eval_token_precision_weighted': 0.9599372218014981, 'eval_token_recall_weighted': 0.9583892538437992, 'eval_token_f1_weighted': 0.9584441076581565, 'eval_ai_token_precision': 0.9298781039358751, 'eval_ai_token_recall': 0.9848100675574918, 'eval_ai_token_f1': 0.9565560930405466, 'eval_seqeval_precision': 0.2866554834317944, 'eval_seqeval_recall': 0.80815627141878, 'eval_seqeval_f1': 0.423200473772051, 'eval_class_O_precision': 0.9861633262856427, 'eval_class_O_recall': 0.935806206633163, 'eval_class_O_f1': 0.960325068306997, 'eval_class_B-AI_precision': 0.8340403315277687, 'eval_class_B-AI_recall': 0.8759424263193969, 'eval_class_B-AI_f1': 0.8544779861598636, 'eval_class_I-AI_precision': 0.929919769969046, 'eval_class_I-AI_recall': 0.9848819573894063, 'eval_class_I-AI_f1': 0.9566120504034521, 'eval_runtime': 579.6


100%|██████████| 13165/13165 [09:29<00:00, 23.10it/s]

[TRAIN] Evaluating on test...



100%|██████████| 14159/14159 [10:14<00:00, 23.04it/s]

[TRAIN] Building classification reports...



100%|██████████| 14159/14159 [10:14<00:00, 23.05it/s]


[TRAIN] Saving final model + tokenizer...

=== TRAINING FINISHED ===
Final model: /home/jupyter/project/ai_detector_diploma/src/token_detector/token_detector_block2/2d_full_finetune/final_model
Training summary: /home/jupyter/project/ai_detector_diploma/src/token_detector/token_detector_block2/2d_full_finetune/training_summary.json

Validation metrics:
{
  "val_loss": 0.13609594106674194,
  "val_token_precision_macro": 0.9167078092608193,
  "val_token_recall_macro": 0.9322101967806553,
  "val_token_f1_macro": 0.9238050349567709,
  "val_token_precision_weighted": 0.9599372218014981,
  "val_token_recall_weighted": 0.9583892538437992,
  "val_token_f1_weighted": 0.9584441076581565,
  "val_ai_token_precision": 0.9298781039358751,
  "val_ai_token_recall": 0.9848100675574918,
  "val_ai_token_f1": 0.9565560930405466,
  "val_seqeval_precision": 0.2866554834317944,
  "val_seqeval_recall": 0.80815627141878,
  "val_seqeval_f1": 0.423200473772051,
  "val_class_O_precision": 0.9861633262856427,
  "v

In [9]:
import json
from pathlib import Path
import pandas as pd

# Укажи свою папку с экспериментами Block 2
BLOCK2_OUT = Path("token_detector_block2")   # если нужно, поправь путь

if not BLOCK2_OUT.exists():
    raise FileNotFoundError(f"Directory not found: {BLOCK2_OUT.resolve()}")

rows = []

# Берём все подпапки внутри block2
experiment_dirs = sorted([p for p in BLOCK2_OUT.iterdir() if p.is_dir()])

print("Found experiment directories:")
for p in experiment_dirs:
    print("-", p.name)

for exp_dir in experiment_dirs:
    run_metadata_path = exp_dir / "run_metadata.json"
    val_metrics_path = exp_dir / "val_metrics.json"
    test_metrics_path = exp_dir / "test_metrics.json"

    row = {
        "experiment": exp_dir.name,
        "output_dir": str(exp_dir),
    }

    # run_metadata
    if run_metadata_path.exists():
        with open(run_metadata_path, "r", encoding="utf-8") as f:
            run_meta = json.load(f)

        run_cfg = run_meta.get("run_config", {})
        row["train_mode"] = run_cfg.get("train_mode")
        row["partial_unfreeze_last_n_layers"] = run_cfg.get("partial_unfreeze_last_n_layers")
        row["learning_rate"] = run_cfg.get("learning_rate")
        row["num_train_epochs"] = run_cfg.get("num_train_epochs")
        row["gradient_accumulation_steps"] = run_cfg.get("gradient_accumulation_steps")
        row["trainable_params"] = run_meta.get("trainable_params")
        row["dataset_source"] = run_meta.get("dataset_source")

    # val_metrics
    if val_metrics_path.exists():
        with open(val_metrics_path, "r", encoding="utf-8") as f:
            val_metrics = json.load(f)

        row["val_seqeval_f1"] = val_metrics.get("val_seqeval_f1")
        row["val_ai_token_f1"] = val_metrics.get("val_ai_token_f1")
        row["val_class_B-AI_f1"] = val_metrics.get("val_class_B-AI_f1")
        row["val_class_O_recall"] = val_metrics.get("val_class_O_recall")
        row["val_token_f1_macro"] = val_metrics.get("val_token_f1_macro")
        row["val_loss"] = val_metrics.get("val_loss")

    # test_metrics
    if test_metrics_path.exists():
        with open(test_metrics_path, "r", encoding="utf-8") as f:
            test_metrics = json.load(f)

        row["test_seqeval_f1"] = test_metrics.get("test_seqeval_f1")
        row["test_ai_token_f1"] = test_metrics.get("test_ai_token_f1")
        row["test_class_B-AI_f1"] = test_metrics.get("test_class_B-AI_f1")
        row["test_class_O_recall"] = test_metrics.get("test_class_O_recall")
        row["test_token_f1_macro"] = test_metrics.get("test_token_f1_macro")
        row["test_loss"] = test_metrics.get("test_loss")

    rows.append(row)

df = pd.DataFrame(rows)

if df.empty:
    raise RuntimeError(f"No experiment results found in {BLOCK2_OUT.resolve()}")

# Сортировка по главной метрике
if "test_seqeval_f1" in df.columns:
    df_sorted = df.sort_values("test_seqeval_f1", ascending=False, na_position="last")
else:
    df_sorted = df.copy()

print("\n=== BLOCK 2 RESULTS ===")
print(df_sorted.to_string(index=False))

# сохранить csv
csv_path = BLOCK2_OUT / "block2_results_from_disk.csv"
df_sorted.to_csv(csv_path, index=False)
print(f"\nSaved CSV to: {csv_path.resolve()}")

# выбрать победителя
if "test_seqeval_f1" in df_sorted.columns and df_sorted["test_seqeval_f1"].notna().any():
    best = df_sorted.loc[df_sorted["test_seqeval_f1"].idxmax()]

    print("\n🏆 BEST EXPERIMENT IN BLOCK 2")
    print(f"experiment:                {best.get('experiment', None)}")
    print(f"train_mode:                {best.get('train_mode', None)}")
    print(f"partial_unfreeze_layers:   {best.get('partial_unfreeze_last_n_layers', None)}")
    print(f"learning_rate:             {best.get('learning_rate', None)}")
    print(f"num_train_epochs:          {best.get('num_train_epochs', None)}")
    print(f"trainable_params:          {best.get('trainable_params', None)}")
    print(f"test_seqeval_f1:           {best.get('test_seqeval_f1', None)}")
    print(f"test_ai_token_f1:          {best.get('test_ai_token_f1', None)}")
    print(f"test_class_B-AI_f1:        {best.get('test_class_B-AI_f1', None)}")
    print(f"test_class_O_recall:       {best.get('test_class_O_recall', None)}")
    print(f"test_token_f1_macro:       {best.get('test_token_f1_macro', None)}")
    print(f"test_loss:                 {best.get('test_loss', None)}")
else:
    print("\nNo valid test_seqeval_f1 found to select a winner.")

Found experiment directories:
- 2a_head_only
- 2b_partial_unfreeze_last_2
- 2c_partial_unfreeze_last_4
- 2d_full_finetune

=== BLOCK 2 RESULTS ===
                experiment                                       output_dir       train_mode  partial_unfreeze_last_n_layers  learning_rate  num_train_epochs  gradient_accumulation_steps  trainable_params   dataset_source  val_seqeval_f1  val_ai_token_f1  val_class_B-AI_f1  val_class_O_recall  val_token_f1_macro  val_loss  test_seqeval_f1  test_ai_token_f1  test_class_B-AI_f1  test_class_O_recall  test_token_f1_macro  test_loss
          2d_full_finetune           token_detector_block2/2d_full_finetune             full                               2       0.000001                 1                            4         278221059 dataset_override        0.423200         0.956556           0.854478            0.935806            0.923805  0.136096         0.415731          0.954522            0.853179             0.929622             0.921603 

In [4]:
import json
from pathlib import Path
import pandas as pd

from train_token_classifier import run_training, TrainConfig

WORKDIR = Path.cwd()

BLOCK3_OUT = WORKDIR / "token_detector_block3"
BLOCK3_OUT.mkdir(parents=True, exist_ok=True)

# Победитель Block 2 — full fine-tuning model init
BEST_BLOCK2_MODEL = "microsoft/mdeberta-v3-base"

print("WORKDIR:", WORKDIR)
print("BLOCK3_OUT:", BLOCK3_OUT)

# Базовые параметры — фиксируем лучший режим из Block 2
base_config = dict(
    hf_dataset_dir="",
    bio_data_dir="",

    model_name=BEST_BLOCK2_MODEL,
    train_mode="full",
    partial_unfreeze_last_n_layers=2,   # не используется в full, но пусть поле остаётся

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,

    learning_rate=1e-6,
    weight_decay=0.01,
    warmup_ratio=0.03,
    max_grad_norm=0.5,

    logging_steps=50,
    save_total_limit=2,
    dataloader_num_workers=2,
    seed=42,

    fp16=False,
    bf16=False,
    cpu_only=False,
    gradient_checkpointing=False,
    group_by_length=True,

    # Тот же loss setup, что и у победителя Block 2
    use_focal_loss=False,
    use_class_balanced_alpha=False,
    manual_alpha=[1.0, 3.0, 5.0],
    effective_num_beta=0.9999,
    alpha_clip_max=10.0,
    focal_gamma=1.5,

    use_weighted_sampler=False,
    sampler_human_weight=1.0,
    sampler_ai_weight=1.5,
    sampler_bai_weight=4.0,

    # Теперь включаем сохранение лучшего чекпоинта
    eval_strategy="epoch",
    save_strategy="epoch",
    eval_steps=500,
    save_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model="eval_seqeval_f1",
    greater_is_better=True,
)

experiments = {
    "3a_full_e1": {
        "num_train_epochs": 1,
    },
    "3b_full_e2": {
        "num_train_epochs": 2,
    },
    "3c_full_e3": {
        "num_train_epochs": 3,
    },
    "3d_full_e5": {
        "num_train_epochs": 5,
    },
}


WORKDIR: /home/jupyter/project/ai_detector_diploma/src/token_detector
BLOCK3_OUT: /home/jupyter/project/ai_detector_diploma/src/token_detector/token_detector_block3


In [ ]:
results = []

for exp_name, exp_params in experiments.items():
    output_dir = BLOCK3_OUT / exp_name

    print(f"\n{'=' * 80}")
    print(f"RUNNING EXPERIMENT: {exp_name}")
    print(f"OUTPUT DIR: {output_dir}")
    print(f"{'=' * 80}")

    cfg_dict = {**base_config, **exp_params, "output_dir": str(output_dir)}
    cfg = TrainConfig(**cfg_dict)

    run_training(cfg, dataset_override=ds)

    test_metrics_path = output_dir / "test_metrics.json"
    val_metrics_path = output_dir / "val_metrics.json"
    run_metadata_path = output_dir / "run_metadata.json"
    train_metrics_path = output_dir / "train_metrics.json"

    row = {
        "experiment": exp_name,
        "output_dir": str(output_dir),
    }

    if run_metadata_path.exists():
        with open(run_metadata_path, "r", encoding="utf-8") as f:
            run_meta = json.load(f)

        rcfg = run_meta["run_config"]
        row["train_mode"] = rcfg.get("train_mode")
        row["learning_rate"] = rcfg.get("learning_rate")
        row["num_train_epochs"] = rcfg.get("num_train_epochs")
        row["trainable_params"] = run_meta.get("trainable_params")

    if train_metrics_path.exists():
        with open(train_metrics_path, "r", encoding="utf-8") as f:
            train_metrics = json.load(f)
        row["train_loss"] = train_metrics.get("train_loss")

    if val_metrics_path.exists():
        with open(val_metrics_path, "r", encoding="utf-8") as f:
            val_metrics = json.load(f)

        row["val_seqeval_f1"] = val_metrics.get("val_seqeval_f1")
        row["val_ai_token_f1"] = val_metrics.get("val_ai_token_f1")
        row["val_class_B-AI_f1"] = val_metrics.get("val_class_B-AI_f1")
        row["val_class_O_recall"] = val_metrics.get("val_class_O_recall")
        row["val_token_f1_macro"] = val_metrics.get("val_token_f1_macro")
        row["val_loss"] = val_metrics.get("val_loss")

    if test_metrics_path.exists():
        with open(test_metrics_path, "r", encoding="utf-8") as f:
            test_metrics = json.load(f)

        row["test_seqeval_f1"] = test_metrics.get("test_seqeval_f1")
        row["test_ai_token_f1"] = test_metrics.get("test_ai_token_f1")
        row["test_class_B-AI_f1"] = test_metrics.get("test_class_B-AI_f1")
        row["test_class_O_recall"] = test_metrics.get("test_class_O_recall")
        row["test_token_f1_macro"] = test_metrics.get("test_token_f1_macro")
        row["test_loss"] = test_metrics.get("test_loss")

    results.append(row)

results_path = BLOCK3_OUT / "block3_results.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\nSaved raw results to: {results_path}")


 57%|█████▋    | 16000/28201 [1:55:51<1:20:21,  2.53it/s]

{'loss': 0.1162, 'grad_norm': 0.08502154052257538, 'learning_rate': 4.460407984207063e-07, 'epoch': 0.57}


 57%|█████▋    | 16050/28201 [1:56:13<1:19:50,  2.54it/s]

{'loss': 0.1486, 'grad_norm': 24.02999496459961, 'learning_rate': 4.442129121883454e-07, 'epoch': 0.57}


 57%|█████▋    | 16100/28201 [1:56:35<1:19:24,  2.54it/s]

...

 66%|██████▋   | 18700/28201 [2:15:23<1:02:01,  2.55it/s]

{'loss': 0.1362, 'grad_norm': 0.27434781193733215, 'learning_rate': 3.4733494187321776e-07, 'epoch': 0.66}


 66%|██████▋   | 18750/28201 [2:15:45<1:01:52,  2.55it/s]

{'loss': 0.1332, 'grad_norm': 75.43495178222656, 'learning_rate': 3.455070556408569e-07, 'epoch': 0.66}


 66%|██████▋   | 18753/28201 [2:15:46<1:07:06,  2.35it/s]

In [ ]:
df = pd.DataFrame(results)

# Сортируем по главной span-level метрике
df_sorted = df.sort_values("test_seqeval_f1", ascending=False, na_position="last")

print("\n=== BLOCK 3 RESULTS ===")
print(df_sorted.to_string(index=False))

csv_path = BLOCK3_OUT / "block3_results.csv"
df_sorted.to_csv(csv_path, index=False)
print(f"\nSaved CSV to: {csv_path}")

if "test_seqeval_f1" in df_sorted.columns and df_sorted["test_seqeval_f1"].notna().any():
    best = df_sorted.loc[df_sorted["test_seqeval_f1"].idxmax()]

    print("\n🏆 BEST EXPERIMENT IN BLOCK 3")
    print(f"experiment:                {best.get('experiment', None)}")
    print(f"train_mode:                {best.get('train_mode', None)}")
    print(f"learning_rate:             {best.get('learning_rate', None)}")
    print(f"num_train_epochs:          {best.get('num_train_epochs', None)}")
    print(f"trainable_params:          {best.get('trainable_params', None)}")
    print(f"train_loss:                {best.get('train_loss', None)}")
    print(f"val_loss:                  {best.get('val_loss', None)}")
    print(f"test_loss:                 {best.get('test_loss', None)}")
    print(f"test_seqeval_f1:           {best.get('test_seqeval_f1', None)}")
    print(f"test_ai_token_f1:          {best.get('test_ai_token_f1', None)}")
    print(f"test_class_B-AI_f1:        {best.get('test_class_B-AI_f1', None)}")
    print(f"test_class_O_recall:       {best.get('test_class_O_recall', None)}")
    print(f"test_token_f1_macro:       {best.get('test_token_f1_macro', None)}")
else:
    print("No valid results found in Block 3.")


In [2]:
import json
from pathlib import Path

from train_token_classifier import run_training, TrainConfig

WORKDIR = Path.cwd()

BLOCK3_OUT = WORKDIR / "token_detector_block3"
BLOCK3_OUT.mkdir(parents=True, exist_ok=True)

# Победитель Block 2
BEST_BLOCK2_MODEL = WORKDIR / "token_detector_block2" / "2d_full_finetune" / "final_model"

print("WORKDIR:", WORKDIR)
print("BLOCK3_OUT:", BLOCK3_OUT)
print("BEST_BLOCK2_MODEL exists:", BEST_BLOCK2_MODEL.exists())

# Один long-run вместо 4 отдельных запусков
LONGRUN_OUT = BLOCK3_OUT / "3_full_longrun_e5"

config = TrainConfig(
    hf_dataset_dir="",
    bio_data_dir="",
    model_name=str(BEST_BLOCK2_MODEL),
    output_dir=str(LONGRUN_OUT),

    train_mode="full",
    partial_unfreeze_last_n_layers=2,   # не используется в full, но поле оставляем

    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,

    learning_rate=1e-6,
    weight_decay=0.01,
    warmup_ratio=0.03,
    max_grad_norm=0.5,

    logging_steps=50,
    save_total_limit=6,       # важно: чтобы сохранить эпохи
    dataloader_num_workers=2,
    seed=42,

    fp16=False,
    bf16=False,
    cpu_only=False,
    gradient_checkpointing=False,
    group_by_length=True,

    # Победившая схема из Block 2
    use_focal_loss=False,
    use_class_balanced_alpha=False,
    manual_alpha=[1.0, 3.0, 5.0],
    effective_num_beta=0.9999,
    alpha_clip_max=10.0,
    focal_gamma=1.5,

    use_weighted_sampler=False,
    sampler_human_weight=1.0,
    sampler_ai_weight=1.5,
    sampler_bai_weight=4.0,

    # Ключевое для long-run
    eval_strategy="epoch",
    save_strategy="epoch",
    eval_steps=500,
    save_steps=500,

    # Для исследования по эпохам удобнее сначала НЕ загружать best model автоматически
    load_best_model_at_end=False,
)

run_training(config, dataset_override=ds)


 91%|█████████ | 128300/141005 [16:21:59<1:27:41,  2.41it/s]

{'loss': 0.0388, 'grad_norm': 0.25068792700767517, 'learning_rate': 9.289046163744571e-08, 'epoch': 4.55}


 91%|█████████ | 128350/141005 [16:22:20<1:26:21,  2.44it/s]

{'loss': 0.0714, 'grad_norm': 0.8775083422660828, 'learning_rate': 9.25248950823987e-08, 'epoch': 4.55}


 91%|█████████ | 128400/141005 [16:22:42<1:26:46,  2.42it/s]

...

[TRAIN] Evaluating on test...



100%|██████████| 14159/14159 [10:02<00:00, 23.49it/s]

[TRAIN] Building classification reports...



100%|██████████| 14159/14159 [10:02<00:00, 23.49it/s]


[TRAIN] Saving final model + tokenizer...

=== TRAINING FINISHED ===
Final model: /home/jupyter/project/ai_detector_diploma/src/token_detector/token_detector_block3/3_full_longrun_e5/final_model
Training summary: /home/jupyter/project/ai_detector_diploma/src/token_detector/token_detector_block3/3_full_longrun_e5/training_summary.json

Validation metrics:
{
  "val_loss": 0.11703871935606003,
  "val_token_precision_macro": 0.9346193641388609,
  "val_token_recall_macro": 0.9535194934730628,
  "val_token_f1_macro": 0.9435987169361869,
  "val_token_precision_weighted": 0.9720506963603944,
  "val_token_recall_weighted": 0.971371588313737,
  "val_token_f1_weighted": 0.9714028628965026,
  "val_ai_token_precision": 0.9518817731975172,
  "val_ai_token_recall": 0.988428246076767,
  "val_ai_token_f1": 0.9698108263193614,
  "val_seqeval_precision": 0.5247054905715356,
  "val_seqeval_recall": 0.8639479095270733,
  "val_seqeval_f1": 0.6528889234195736,
  "val_class_O_precision": 0.9896540614861918,
 

In [3]:
import json
from pathlib import Path
import pandas as pd

LONGRUN_OUT = Path("token_detector_block3") / "3_full_longrun_e5"

# Ищем trainer_state.json
candidate_paths = [
    LONGRUN_OUT / "checkpoints" / "trainer_state.json",
    LONGRUN_OUT / "trainer_state.json",
]

trainer_state_path = None
for p in candidate_paths:
    if p.exists():
        trainer_state_path = p
        break

if trainer_state_path is None:
    # fallback: ищем по checkpoint-* подпапкам
    checkpoint_dirs = sorted((LONGRUN_OUT / "checkpoints").glob("checkpoint-*")) if (LONGRUN_OUT / "checkpoints").exists() else []
    for ckpt in checkpoint_dirs[::-1]:
        p = ckpt / "trainer_state.json"
        if p.exists():
            trainer_state_path = p
            break

if trainer_state_path is None:
    raise FileNotFoundError("trainer_state.json not found. Проверь, что training script сохраняет checkpoints.")

print("Using trainer_state:", trainer_state_path)

with open(trainer_state_path, "r", encoding="utf-8") as f:
    trainer_state = json.load(f)

log_history = trainer_state.get("log_history", [])
print(f"log_history entries: {len(log_history)}")

epoch_rows = []

for row in log_history:
    # Берём только eval-строки по эпохам
    if "epoch" not in row:
        continue

    # интересуют строки, где есть eval-метрики
    has_eval = any(k.startswith("eval_") for k in row.keys())
    if not has_eval:
        continue

    epoch_rows.append({
        "epoch": row.get("epoch"),
        "eval_loss": row.get("eval_loss"),
        "eval_seqeval_f1": row.get("eval_seqeval_f1"),
        "eval_seqeval_precision": row.get("eval_seqeval_precision"),
        "eval_seqeval_recall": row.get("eval_seqeval_recall"),
        "eval_ai_token_f1": row.get("eval_ai_token_f1"),
        "eval_class_B-AI_f1": row.get("eval_class_B-AI_f1"),
        "eval_class_O_recall": row.get("eval_class_O_recall"),
        "eval_token_f1_macro": row.get("eval_token_f1_macro"),
        "step": row.get("step"),
    })

df_epochs = pd.DataFrame(epoch_rows)

if df_epochs.empty:
    raise RuntimeError("Не удалось извлечь eval-метрики по эпохам из trainer_state.json")

df_epochs = df_epochs.sort_values("epoch").reset_index(drop=True)

print("\n=== BLOCK 3 EPOCH TABLE ===")
print(df_epochs.to_string(index=False))

csv_path = LONGRUN_OUT / "block3_epoch_table.csv"
df_epochs.to_csv(csv_path, index=False)
print(f"\nSaved epoch table to: {csv_path.resolve()}")


Using trainer_state: token_detector_block3/3_full_longrun_e5/checkpoints/checkpoint-84603/trainer_state.json
log_history entries: 1695

=== BLOCK 3 EPOCH TABLE ===
 epoch  eval_loss  eval_seqeval_f1  eval_seqeval_precision  eval_seqeval_recall  eval_ai_token_f1  eval_class_B-AI_f1  eval_class_O_recall  eval_token_f1_macro  step
   1.0   0.125051         0.543066                0.401855             0.837286          0.962400            0.870551             0.945253             0.932995 28201
   2.0   0.104322         0.618155                0.485468             0.850651          0.970379            0.886987             0.960036             0.943682 56402
   3.0   0.098683         0.671986                0.551386             0.860110          0.974162            0.897055             0.966476             0.949470 84603

Saved epoch table to: /home/jupyter/project/ai_detector_diploma/src/token_detector/token_detector_block3/3_full_longrun_e5/block3_epoch_table.csv


In [4]:
import pandas as pd
from pathlib import Path

LONGRUN_OUT = Path("token_detector_block3") / "3_full_longrun_e5"
df_epochs = pd.read_csv(LONGRUN_OUT / "block3_epoch_table.csv")

# Главная метрика — seqeval_f1
best_idx = df_epochs["eval_seqeval_f1"].idxmax()
best = df_epochs.loc[best_idx]

print("🏆 BEST EPOCH BY eval_seqeval_f1")
print(f"epoch:                 {best['epoch']}")
print(f"eval_loss:             {best['eval_loss']}")
print(f"eval_seqeval_f1:       {best['eval_seqeval_f1']}")
print(f"eval_seqeval_precision:{best['eval_seqeval_precision']}")
print(f"eval_seqeval_recall:   {best['eval_seqeval_recall']}")
print(f"eval_ai_token_f1:      {best['eval_ai_token_f1']}")
print(f"eval_class_B-AI_f1:    {best['eval_class_B-AI_f1']}")
print(f"eval_class_O_recall:   {best['eval_class_O_recall']}")
print(f"eval_token_f1_macro:   {best['eval_token_f1_macro']}")
print(f"step:                  {best['step']}")

# Дополнительно можно посмотреть топ-3 эпох
print("\n=== TOP 3 EPOCHS BY eval_seqeval_f1 ===")
print(
    df_epochs
    .sort_values("eval_seqeval_f1", ascending=False)
    .head(3)
    .to_string(index=False)
)


🏆 BEST EPOCH BY eval_seqeval_f1
epoch:                 3.0
eval_loss:             0.0986833646893501
eval_seqeval_f1:       0.6719858630753167
eval_seqeval_precision:0.551386264774375
eval_seqeval_recall:   0.8601096641535299
eval_ai_token_f1:      0.9741618975897608
eval_class_B-AI_f1:    0.897055359246172
eval_class_O_recall:   0.9664758163351228
eval_token_f1_macro:   0.9494696532746292
step:                  84603.0

=== TOP 3 EPOCHS BY eval_seqeval_f1 ===
 epoch  eval_loss  eval_seqeval_f1  eval_seqeval_precision  eval_seqeval_recall  eval_ai_token_f1  eval_class_B-AI_f1  eval_class_O_recall  eval_token_f1_macro  step
   3.0   0.098683         0.671986                0.551386             0.860110          0.974162            0.897055             0.966476             0.949470 84603
   2.0   0.104322         0.618155                0.485468             0.850651          0.970379            0.886987             0.960036             0.943682 56402
   1.0   0.125051         0.543066   

In [1]:
from pathlib import Path

LONGRUN_OUT = Path("token_detector_block3") / "3_full_longrun_e5"
ckpt_dir = LONGRUN_OUT / "checkpoints"

print("Checkpoint dir exists:", ckpt_dir.exists())
if ckpt_dir.exists():
    ckpts = sorted([p for p in ckpt_dir.iterdir() if p.is_dir()])
    print("Checkpoint folders:")
    for p in ckpts:
        print("-", p.name)

Checkpoint dir exists: True
Checkpoint folders:
- checkpoint-112804
- checkpoint-141005
- checkpoint-28201
- checkpoint-56402
- checkpoint-84603


In [2]:
import json
from pathlib import Path
import pandas as pd

LONGRUN_OUT = Path("token_detector_block3") / "3_full_longrun_e5"
CKPT_DIR = LONGRUN_OUT / "checkpoints"

if not CKPT_DIR.exists():
    raise FileNotFoundError(f"Checkpoint dir not found: {CKPT_DIR.resolve()}")

checkpoint_dirs = sorted(
    [p for p in CKPT_DIR.iterdir() if p.is_dir() and p.name.startswith("checkpoint-")],
    key=lambda p: int(p.name.split("-")[-1])
)

rows = []

for ckpt in checkpoint_dirs:
    trainer_state_path = ckpt / "trainer_state.json"
    if not trainer_state_path.exists():
        print(f"WARNING: trainer_state.json missing in {ckpt}")
        continue

    with open(trainer_state_path, "r", encoding="utf-8") as f:
        trainer_state = json.load(f)

    log_history = trainer_state.get("log_history", [])

    # Берём последнюю eval-строку в этом checkpoint
    eval_rows = [
        row for row in log_history
        if "epoch" in row and any(k.startswith("eval_") for k in row.keys())
    ]

    if not eval_rows:
        print(f"WARNING: no eval rows found in {trainer_state_path}")
        continue

    row = eval_rows[-1]

    rows.append({
        "checkpoint": ckpt.name,
        "step": int(ckpt.name.split("-")[-1]),
        "epoch": row.get("epoch"),

        "eval_loss": row.get("eval_loss"),
        "eval_seqeval_f1": row.get("eval_seqeval_f1"),
        "eval_seqeval_precision": row.get("eval_seqeval_precision"),
        "eval_seqeval_recall": row.get("eval_seqeval_recall"),

        "eval_ai_token_f1": row.get("eval_ai_token_f1"),
        "eval_class_B-AI_f1": row.get("eval_class_B-AI_f1"),
        "eval_class_O_recall": row.get("eval_class_O_recall"),
        "eval_token_f1_macro": row.get("eval_token_f1_macro"),
    })

df_epochs = pd.DataFrame(rows).sort_values("epoch").reset_index(drop=True)

print("\n=== BLOCK 3 EPOCH TABLE (from checkpoints) ===")
print(df_epochs.to_string(index=False))

csv_path = LONGRUN_OUT / "block3_epoch_table_from_checkpoints.csv"
df_epochs.to_csv(csv_path, index=False)
print(f"\nSaved epoch table to: {csv_path.resolve()}")



=== BLOCK 3 EPOCH TABLE (from checkpoints) ===
       checkpoint   step  epoch  eval_loss  eval_seqeval_f1  eval_seqeval_precision  eval_seqeval_recall  eval_ai_token_f1  eval_class_B-AI_f1  eval_class_O_recall  eval_token_f1_macro
 checkpoint-28201  28201    1.0   0.125051         0.543066                0.401855             0.837286          0.962400            0.870551             0.945253             0.932995
 checkpoint-56402  56402    2.0   0.104322         0.618155                0.485468             0.850651          0.970379            0.886987             0.960036             0.943682
 checkpoint-84603  84603    3.0   0.098683         0.671986                0.551386             0.860110          0.974162            0.897055             0.966476             0.949470
checkpoint-112804 112804    4.0   0.113148         0.663662                0.539110             0.863057          0.971022            0.890296             0.959296             0.945168
checkpoint-141005 141005   

In [3]:
import pandas as pd
from pathlib import Path

LONGRUN_OUT = Path("token_detector_block3") / "3_full_longrun_e5"
df_epochs = pd.read_csv(LONGRUN_OUT / "block3_epoch_table_from_checkpoints.csv")

best_idx = df_epochs["eval_seqeval_f1"].idxmax()
best = df_epochs.loc[best_idx]

print("🏆 BEST EPOCH BY eval_seqeval_f1")
print(f"checkpoint:              {best['checkpoint']}")
print(f"epoch:                   {best['epoch']}")
print(f"step:                    {best['step']}")
print(f"eval_loss:               {best['eval_loss']}")
print(f"eval_seqeval_f1:         {best['eval_seqeval_f1']}")
print(f"eval_seqeval_precision:  {best['eval_seqeval_precision']}")
print(f"eval_seqeval_recall:     {best['eval_seqeval_recall']}")
print(f"eval_ai_token_f1:        {best['eval_ai_token_f1']}")
print(f"eval_class_B-AI_f1:      {best['eval_class_B-AI_f1']}")
print(f"eval_class_O_recall:     {best['eval_class_O_recall']}")
print(f"eval_token_f1_macro:     {best['eval_token_f1_macro']}")

print("\n=== ALL EPOCHS SORTED BY eval_seqeval_f1 ===")
print(
    df_epochs.sort_values("eval_seqeval_f1", ascending=False).to_string(index=False)
)


🏆 BEST EPOCH BY eval_seqeval_f1
checkpoint:              checkpoint-84603
epoch:                   3.0
step:                    84603
eval_loss:               0.0986833646893501
eval_seqeval_f1:         0.6719858630753167
eval_seqeval_precision:  0.551386264774375
eval_seqeval_recall:     0.8601096641535299
eval_ai_token_f1:        0.9741618975897608
eval_class_B-AI_f1:      0.897055359246172
eval_class_O_recall:     0.9664758163351228
eval_token_f1_macro:     0.9494696532746292

=== ALL EPOCHS SORTED BY eval_seqeval_f1 ===
       checkpoint   step  epoch  eval_loss  eval_seqeval_f1  eval_seqeval_precision  eval_seqeval_recall  eval_ai_token_f1  eval_class_B-AI_f1  eval_class_O_recall  eval_token_f1_macro
 checkpoint-84603  84603    3.0   0.098683         0.671986                0.551386             0.860110          0.974162            0.897055             0.966476             0.949470
checkpoint-112804 112804    4.0   0.113148         0.663662                0.539110             0.86

In [4]:
import json
import shutil
from pathlib import Path
import pandas as pd

LONGRUN_OUT = Path("token_detector_block3") / "3_full_longrun_e5"
epoch_table_path = LONGRUN_OUT / "block3_epoch_table_from_checkpoints.csv"

df_epochs = pd.read_csv(epoch_table_path)

best_idx = df_epochs["eval_seqeval_f1"].idxmax()
best = df_epochs.loc[best_idx]

best_checkpoint_name = best["checkpoint"]
best_epoch = best["epoch"]

best_checkpoint_dir = LONGRUN_OUT / "checkpoints" / best_checkpoint_name
best_model_dir = LONGRUN_OUT / "best_model_epoch3"

print("Best checkpoint:", best_checkpoint_dir)
print("Best epoch:", best_epoch)
print("Exists:", best_checkpoint_dir.exists())

# Если папка уже есть — удалим и пересоздадим
if best_model_dir.exists():
    shutil.rmtree(best_model_dir)

shutil.copytree(best_checkpoint_dir, best_model_dir)

print("Copied best checkpoint to:", best_model_dir.resolve())


Best checkpoint: token_detector_block3/3_full_longrun_e5/checkpoints/checkpoint-84603
Best epoch: 3.0
Exists: True
Copied best checkpoint to: /home/jupyter/project/ai_detector_diploma/src/token_detector/token_detector_block3/3_full_longrun_e5/best_model_epoch3


In [5]:
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForTokenClassification

BEST_MODEL_DIR = Path("token_detector_block3") / "3_full_longrun_e5" / "best_model_epoch3"

tokenizer = AutoTokenizer.from_pretrained(str(BEST_MODEL_DIR), use_fast=True)
model = AutoModelForTokenClassification.from_pretrained(str(BEST_MODEL_DIR))

print("Loaded tokenizer from:", BEST_MODEL_DIR)
print("Loaded model num_labels:", model.config.num_labels)
print("id2label:", model.config.id2label)
print("label2id:", model.config.label2id)


/home/jupyter/.local/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Loaded tokenizer from: token_detector_block3/3_full_longrun_e5/best_model_epoch3
Loaded model num_labels: 3
id2label: {0: 'O', 1: 'B-AI', 2: 'I-AI'}
label2id: {'B-AI': 1, 'I-AI': 2, 'O': 0}
